In [7]:
!pip install lifetimes pandas

In [8]:
pip install --upgrade pip

  Using cached pip-26.2.1-py3-none-any.whl.metadata (4.6 kB)
Using cached pip-26.2.1-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 26.1.2
    Uninstalling pip-26.1.2:
      Successfully uninstalled pip-26.1.2
Note: you may need to restart the kernel to use updated packages.


In [57]:
import pandas as pd
import warnings
from lifetimes import BetaGeoFitter, GammaGammaFitter

In [58]:
warnings.filterwarnings('ignore')

In [59]:
# 1. Load the data
df = pd.read_csv(r"C:\Users\barath babu\Downloads\DATA ANALYTICS\project-1\python\customer_rfm.csv")
df = df.dropna()
# --- THE FINAL FIX: Remove Same-Day Repeat Purchases ---
# The math cannot handle frequency > 0 when recency == 0 (causes division by zero)
df = df[(df['frequency'] > 0) & (df['recency'] > 0)]

# --- Scale to Weeks (prevents Float Overflow) ---
df['recency'] = df['recency'] / 7.0
df['T'] = df['T'] / 7.0

# Clean any lingering time paradoxes
df = df[(df['T'] > 0) & (df['recency'] <= df['T'])]
# -------------------------------------------------------

In [60]:
# 2. Train the Churn Model (Standard BG/NBD)
bgf = BetaGeoFitter(penalizer_coef=0.01)
bgf.fit(df['frequency'], df['recency'], df['T'])

df['probability_alive'] = bgf.conditional_probability_alive(
    df['frequency'], df['recency'], df['T']
)

# Predict for the next 26 weeks (180 days)
df['predicted_purchases_180_days'] = bgf.predict(
    26, df['frequency'], df['recency'], df['T']
)
print("BG/NBD Churn Model trained successfully!")

BG/NBD Churn Model trained successfully!


In [61]:
# 3. Train the Value Model (Gamma-Gamma)
ggf = GammaGammaFitter(penalizer_coef=0.01)
ggf.fit(df['frequency'], df['monetary_value'])

df['expected_avg_order_value'] = ggf.conditional_expected_average_profit(
    df['frequency'], df['monetary_value']
)

In [62]:
# calculating final lifetime value
df['predicted_LTV_180_days'] = df['predicted_purchases_180_days'] * df['expected_avg_order_value']

In [64]:
# saving file for power bi
df.to_csv(r'C:\Users\barath babu\Downloads\DATA ANALYTICS\project-1\python\final_dashboard_data.csv', index=False)
print("Data processing complete. Your final_dashboard_data.csv is ready for Power BI!")

Data processing complete. Your final_dashboard_data.csv is ready for Power BI!
